# SD4ft-Miner: City vs. Resort cancellations without deposits

We compare cancellation rates between City Hotel and Resort Hotel within the same booking segments.
This task builds on `city_vs_resort_cancellations.ipynb` and adds a condition that removes
bookings with a deposit.

**Question:** In which segments does the City Hotel cancel considerably more often than the
Resort Hotel once the effect of non-refundable deposits is filtered out?

**Hypothesis:** Even among bookings without a deposit there are segments where the City Hotel
cancels at least 1.5 times more often than the Resort Hotel. The overall ratio in these
bookings is only about 1.2.

| Part | Setting |
| --- | --- |
| Antecedent (A) | `market_segment`, `distribution_channel`, `customer_type`, `season` (subset, 1); `lead_time_cat` (seq, 1–2) |
| Antecedent length | 1–2 |
| Succedent (S) | `is_canceled = 1` |
| First set (α) | `hotel = City Hotel` |
| Second set (β) | `hotel = Resort Hotel` |
| Condition (C) | `deposit_type = No Deposit` |
| Quantifiers | `Ratioconf ≥ 1.5`, `Base1 ≥ 150`, `Base2 ≥ 150` |

In [1]:
import pandas as pd
from cleverminer import cleverminer

df = pd.read_csv("../../data/processed/hotel_bookings_categorized.csv", dtype=str)
df = df[[
    "hotel",
    "is_canceled",
    "deposit_type",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "lead_time_cat",
    "season",
]]

print("Bookings:", len(df))

Cleverminer version  1.2.8
Bookings: 118558


## Why the condition on deposit type

Non-refundable bookings are almost always canceled, and almost all of them belong to the City Hotel.
Without the condition, SD4ft-Miner would mostly find subgroups with many non-refundable bookings
instead of real differences between the hotels.

In [2]:
canceled = df["is_canceled"] == "1"

# Number of bookings by deposit type
display(pd.crosstab(df["deposit_type"], df["hotel"]))

# Cancellation rate by deposit type
rates = pd.crosstab(df["deposit_type"], df["hotel"], values=canceled, aggfunc="mean")
display(rates.round(3))

city = rates.loc["No Deposit", "City Hotel"]
resort = rates.loc["No Deposit", "Resort Hotel"]
print(f"No Deposit: City {city:.1%}, Resort {resort:.1%}, ratio {city / resort:.2f}")

hotel,City Hotel,Resort Hotel
deposit_type,,
No Deposit,66007,37803
Non Refund,12867,1719
Refundable,20,142


hotel,City Hotel,Resort Hotel
deposit_type,,
No Deposit,0.306,0.250
Non Refund,0.998,0.960
Refundable,0.700,0.155


No Deposit: City 30.6%, Resort 25.0%, ratio 1.23


## SD4ft-Miner

The condition restricts the whole task to bookings without a deposit.
The threshold `Ratioconf ≥ 1.5` is therefore compared with the overall ratio of 1.23 in these bookings.
`Base1` and `Base2` require at least 150 canceled bookings of the subgroup in each hotel.

In [3]:
miner = cleverminer(
    df=df,
    proc="SD4ftMiner",
    quantifiers={
        "Base1": 150,
        "Base2": 150,
        "Ratioconf": 1.5,
    },
    ante={
        "attributes": [
            {"name": "market_segment", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "distribution_channel", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "customer_type", "type": "subset", "minlen": 1, "maxlen": 1},
            {"name": "lead_time_cat", "type": "seq", "minlen": 1, "maxlen": 2},
            {"name": "season", "type": "subset", "minlen": 1, "maxlen": 1},
        ],
        "minlen": 1,
        "maxlen": 2,
        "type": "con",
    },
    succ={
        "attributes": [
            {"name": "is_canceled", "type": "one", "value": "1"},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    frst={
        "attributes": [
            {"name": "hotel", "type": "one", "value": "City Hotel"},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    scnd={
        "attributes": [
            {"name": "hotel", "type": "one", "value": "Resort Hotel"},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    cond={
        "attributes": [
            {"name": "deposit_type", "type": "one", "value": "No Deposit"},
        ],
        "minlen": 1,
        "maxlen": 1,
        "type": "con",
    },
    opts={
        "no_automatic_data_conversions": True,
        "disable_progressbar": True,
    },
)

Cleverminer version 1.2.8.
Progressbar will not be shown.
No automatic data conversions will be made.
Progressbar will not be shown.
No automatic data conversions will be made.
Starting data preparation ...
Encoding columns into bit-form...
Encoding columns into bit-form...done
Data preparation finished.
Progressbar will not be shown.
No automatic data conversions will be made.
Will go for  SD4ftMiner
Starting to mine rules.
Done. Total verifications : 232, rules 29, times: prep 0.11sec, processing 0.05sec


## Results

Overview of all rules. `RatioConf` is the ratio of the cancellation rates (City / Resort),
`DeltaConf` is their difference.

In [4]:
miner.print_summary()
miner.print_rulelist()


CleverMiner task processing summary:

Task type : SD4ftMiner
Number of verifications : 232
Number of rules : 29
Total time needed : 00h 00m 00s
Time of data preparation : 00h 00m 00s
Time of rule mining : 00h 00m 00s


List of rules:
RULEID BASE1 BASE2 RatioConf DeltaConf Rule
     1   246   170    1.884    +0.052  market_segment(Direct) & lead_time_cat(1_0-7) => is_canceled(1) | deposit_type(No Deposit) : hotel(City Hotel) x hotel(Resort Hotel)
     2   489   366    1.561    +0.051  market_segment(Direct) & lead_time_cat(1_0-7 2_8-30) => is_canceled(1) | deposit_type(No Deposit) : hotel(City Hotel) x hotel(Resort Hotel)
     3  1286   556    1.782    +0.181  market_segment(Groups) & lead_time_cat(4_91-180 5_181-365) => is_canceled(1) | deposit_type(No Deposit) : hotel(City Hotel) x hotel(Resort Hotel)
     4   936   372    2.269    +0.289  market_segment(Groups) & lead_time_cat(5_181-365) => is_canceled(1) | deposit_type(No Deposit) : hotel(City Hotel) x hotel(Resort Hotel)
     5   

### All rules in detail

Fourfold tables of all rules, one below another.

In [5]:
for rule_id in range(1, miner.get_rulecount() + 1):
    miner.print_rule(rule_id)



Rule id : 1

Base1 :   246 Base2 :   170  Relative base 1 : 0.002 Relative base 2 : 0.001 CONF1 : 0.110  CONF2 : +0.059  Delta Conf : +0.052 Ratio Conf : +1.884

Cedents:
  antecedent : market_segment(Direct) & lead_time_cat(1_0-7)
  succcedent : is_canceled(1)
  condition  : deposit_type(No Deposit)
  first set  : hotel(City Hotel)
  second set : hotel(Resort Hotel)

Fourfold tables:
FRST|  S  |  ¬S |  SCND|  S  |  ¬S |
----|-----|-----|  ----|-----|-----| 
 A  |  246| 1982|   A  |  170| 2731|
----|-----|-----|  ----|-----|-----|
¬A  |19958|43821|  ¬A  | 9268|25634|
----|-----|-----|  ----|-----|-----|



Rule id : 2

Base1 :   489 Base2 :   366  Relative base 1 : 0.004 Relative base 2 : 0.003 CONF1 : 0.143  CONF2 : +0.092  Delta Conf : +0.051 Ratio Conf : +1.561

Cedents:
  antecedent : market_segment(Direct) & lead_time_cat(1_0-7 2_8-30)
  succcedent : is_canceled(1)
  condition  : deposit_type(No Deposit)
  first set  : hotel(City Hotel)
  second set : hotel(Resort Hotel)

Fourfo